# Đề 14 – Phát hiện người đi xe máy không đội mũ bảo hiểm (SAM 3.1 + DINOv3)
Chạy trên **Colab T4 / Kaggle T4** (GPU miễn phí). Secrets: đặt `HF_TOKEN` (đã được chấp thuận facebook/sam3.1 và facebook/dinov3-*) và `KAGGLE_API_TOKEN` trong *Colab Secrets* / *Kaggle Secrets* — **không dán token vào notebook**.

In [ ]:
!nvidia-smi
import os
try:
    from google.colab import userdata  # Colab
    for k in ['HF_TOKEN','KAGGLE_API_TOKEN','GH_TOKEN']:
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError:
    from kaggle_secrets import UserSecretsClient  # Kaggle
    s = UserSecretsClient()
    for k in ['HF_TOKEN','KAGGLE_API_TOKEN','GH_TOKEN']:
        try: os.environ[k] = s.get_secret(k)
        except Exception: pass
print({k: bool(os.environ.get(k)) for k in ['HF_TOKEN','KAGGLE_API_TOKEN','GH_TOKEN']})

## 1. Lấy code (repo private → cần GH_TOKEN, hoặc upload zip repo)

In [ ]:
%cd /content
!git clone https://$GH_TOKEN@github.com/meiiie/tgmt-de14-helmet-violation.git de14 || echo 'clone lỗi: upload repo thủ công'
%cd /content/de14
!git clone -q https://github.com/facebookresearch/sam3.git third_party_sam3
!cd third_party_sam3 && git apply ../patches/sam3_cpu.patch && pip -q install --no-deps -e .
!pip -q install ultralytics kaggle timm ftfy regex iopath einops pycocotools scikit-learn joblib lap 'transformers>=4.56'

## 2. Dữ liệu: HELMET test/train clips (annotation Kaggle CC-BY 4.0 theo OSF; frame 640×640 từ HF)

In [ ]:
!bash scripts/get_data.sh
!mkdir -p weights && cd weights && wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt
!python -c "from huggingface_hub import hf_hub_download as h;import shutil;shutil.copy(h('nnsohamnn/helmet-detection-yolo11','yolov11s(80 epochs).pt'),'weights/helmet_yolo11s_nnsohamnn.pt')"

## 3. SAM 3.1 theo frame (prompt motorcycle / person / head) – cache JSON
T4 không hỗ trợ bf16 nhanh/FA3: dùng `--device cuda`, autocast bf16 vẫn chạy (giả lập) — nếu chậm đổi sang fp16 trong `sam31_cache.py`. **FPS T4 là ước lượng tới khi đo.**

In [ ]:
import glob, os, time
clips = ['Bago_highway_10','Bago_highway_20','Bago_rural_3','Bago_rural_49','Bago_urban_3','Bago_urban_50']
for c in clips:
    t=time.time(); !python src/sam31_cache.py data/helmet/frames/{c} --out outputs/sam31/{c}.json --stride 1 --max 100 --device cuda
    print(c, 'sec', time.time()-t)

## 4. DINOv3 head crops + linear probe / kNN (GroupKFold theo clip)

In [ ]:
!python src/build_head_crops.py
!python src/train_probe.py facebook/dinov3-vits16-pretrain-lvd1689m
!python src/train_probe.py facebook/dinov3-vitb16-pretrain-lvd1689m

## 5. Đánh giá theo track: B0 (YOLO helmet từng frame) · B1 (YOLO11+ByteTrack+YOLO helmet) · Core (SAM 3.1 + ByteTrack + DINOv3 probe)

In [ ]:
!python src/eval_helmet.py --method b0
!python src/eval_helmet.py --method b1 --detector yolo --classifier yolo
!python src/eval_helmet.py --method b1 --detector yolo --classifier dino --tag b1_yolo_dinov3
!python src/eval_helmet.py --method b1 --detector sam31 --classifier dino --tag core_sam31_dinov3
!python src/eval_helmet.py --method b1 --detector sam31 --classifier yolo --tag core_sam31_yolohelmet
import json,glob
for f in sorted(glob.glob('outputs/metrics/*.json')): print(f, json.load(open(f)))

## 6. (Tuỳ chọn) SAM 3.1 Object Multiplex – theo vết toàn video bằng text prompt

In [ ]:
import sys; sys.path.insert(0,'src')
from de14.detectors import sam31_video_tracks
tr = sam31_video_tracks('data/helmet/frames/Bago_urban_3', prompts=('motorcycle','person'))
print({p: len({i for f in v.values() for i,_ in f}) for p,v in tr.items()})

## 7. Demo video bất kỳ (vd. clip Hà Nội CC BY-SA 4.0 trên Wikimedia Commons)

In [ ]:
!mkdir -p data/video && wget -q -U 'Mozilla/5.0' -O data/video/hanoi.webm 'https://upload.wikimedia.org/wikipedia/commons/8/8a/Stratkruci%C4%9Do_en_Hanojo.webm'
!python src/run_video.py data/video/hanoi.webm --detector sam31 --classifier dino --stride 3 --out outputs/demo